# M9.2 · NIM Deployment (~20–25 min)

**All steps in this notebook:** pip, GPU, NGC key prompt, catalog NIM pull/run, optional custom Model-Free NIM from M7 SFT.

Production reference: `gsi-training/9.custom_model_deployment/README.md` (8× GPU, TP=2). Workshop: **1× GPU, TP=1**.


## 1. Prerequisites (pip + NGC + NIM)


In [2]:
import sys
from pathlib import Path
import time

# Workshop shared helpers (parent folder: workshop-Materials/).
sys.path.insert(0, str(Path.cwd().parent))
from docker_storage import ensure_docker_storage
from notebook_env import bootstrap_notebook_env, ensure

ensure_docker_storage()  # docker + containerd on /data or /ephemeral (auto-detected)
bootstrap_notebook_env()

# M9 dependencies — installed inline so this notebook is fully self-contained.
for mod, pkg in [
    ("torch", "torch>=2.5.0,<2.7.0"),
    ("transformers", "transformers>=4.45.0,<4.55.0"),
    ("peft", "peft>=0.13.0,<0.16.0"),
    ("requests", "requests>=2.32.0"),
]:
    ensure(mod, [pkg], quiet=True)
print("Prerequisites ready.")

2026-06-16 09:22:58,001 INFO === ensure_docker_storage (storage=/ephemeral, log: /ephemeral/logs/docker_storage.log) ===
2026-06-16 09:22:58,002 INFO disk /: 26.3G used / 96.7G (27.2%)
2026-06-16 09:22:58,002 INFO disk /ephemeral: 99.5G used / 737.2G (600.1G free)
2026-06-16 09:22:58,003 INFO env TMPDIR=/ephemeral/cache/tmp
2026-06-16 09:22:58,003 INFO env DOCKER_TMPDIR=/ephemeral/cache/tmp
2026-06-16 09:22:58,003 INFO env PIP_CACHE_DIR=/ephemeral/cache/pip
2026-06-16 09:22:58,004 INFO env UV_CACHE_DIR=/ephemeral/cache/uv
2026-06-16 09:22:58,004 INFO env HF_HOME=/ephemeral/cache/hf
2026-06-16 09:22:58,004 INFO env XDG_CACHE_HOME=/ephemeral/cache/xdg
2026-06-16 09:22:58,005 INFO env LOCAL_NIM_CACHE=/ephemeral/cache/nim
2026-06-16 09:22:58,005 INFO $ docker info --format {{.DockerRootDir}}
2026-06-16 09:22:58,054 INFO docker data-root (config): /ephemeral/docker
2026-06-16 09:22:58,054 INFO docker data-root (live):   /ephemeral/docker
2026-06-16 09:22:58,055 INFO $ docker info
2026-06-16

docker storage ok: /ephemeral/docker (600.1G free on /ephemeral)
notebook env: /home/shadeform/workshop-materials-gsi/repo-content_v2/workshop-Materials/M9-nvidia_nim/.venv (python /home/shadeform/workshop-materials-gsi/repo-content_v2/workshop-Materials/M9-nvidia_nim/.venv/bin/python, storage /ephemeral)
ok: torch
ok: transformers
ok: peft
ok: requests
Prerequisites ready.


In [3]:
import torch

# Workshop runs on 1x A100 / H100 / H200 — same recipe; verify GPU here.
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    print(f"GPU: {_p.name} ({_p.total_memory / 2**30:.1f} GiB)")
else:
    print("WARNING: No CUDA GPU detected. Training notebooks will not run; Curator small-corpus paths may still work on CPU.")


GPU: NVIDIA H100 PCIe (79.2 GiB)


In [4]:
import os, subprocess
from docker_storage import workshop_nim_cache_dir

_nim_cache = workshop_nim_cache_dir()
subprocess.run(["sudo", "chmod", "777", str(_nim_cache)], check=True)
print(f"NIM cache writable: {_nim_cache}")

NIM cache writable: /ephemeral/cache/nim


In [5]:
import os, subprocess, time, getpass
from pathlib import Path
import requests

# NGC API key: use env var (set before launching Jupyter) or prompt interactively.
if not os.environ.get("NGC_API_KEY"):
    _key = getpass.getpass("Enter your NGC API key: ").strip()
    if _key:
        os.environ["NGC_API_KEY"] = _key
if not os.environ.get("NGC_API_KEY"):
    raise ValueError("NGC_API_KEY is required to pull and run NIM containers from nvcr.io")

LOCAL_NIM_CACHE = Path(os.environ["LOCAL_NIM_CACHE"])  # spacious volume (set by ensure_docker_storage)

NIM_IMAGE = "nvcr.io/nim/nvidia/nvidia-nemotron-nano-9b-v2:latest"
NIM_CONTAINER = "workshop-nim-nano"
NIM_PORT = 8080
NIM_MODEL_ID = "nvidia/nvidia-nemotron-nano-9b-v2"

print("Docker login → nvcr.io …")
subprocess.run(
    ["docker", "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"],
    input=os.environ["NGC_API_KEY"].encode(),
    check=True,
)

# def _nim_ready():
#     try:
#         return requests.get(f"http://localhost:{NIM_PORT}/v1/models", timeout=5).status_code == 200
#     except Exception:
#         return False

# if _nim_ready():
#     print(f"NIM already running on :{NIM_PORT}")
# else:
#     subprocess.run(["docker", "rm", "-f", NIM_CONTAINER], capture_output=True)
#     print("Pulling NIM image (first time: several minutes) …")
#     subprocess.check_call(["docker", "pull", NIM_IMAGE])
#     print("Starting container …")
#     subprocess.check_call([
#         "docker", "run", "-d", "--name", NIM_CONTAINER,
#         "--gpus", "device=0", "--shm-size", "16g",
#         "-e", "NGC_API_KEY", "-e", "NIM_SERVER_PORT=8000",
#         "-v", f"{LOCAL_NIM_CACHE}:/opt/nim/.cache",
#         "-p", f"{NIM_PORT}:8000", NIM_IMAGE,
#     ], env=os.environ.copy())
#     for _i in range(40):
#         if _nim_ready():
#             _r = requests.get(f"http://localhost:{NIM_PORT}/v1/models", timeout=10)
#             print("Models:", [m["id"] for m in _r.json().get("data", [])])
#             break
#         print(f"  waiting for NIM … ({_i + 1}/40)")
#         time.sleep(15)
#     else:
#         raise RuntimeError("NIM not ready — check: docker logs workshop-nim-nano")

# os.environ["LOCAL_NIM_URL"] = f"http://localhost:{NIM_PORT}/v1"
# print("OpenAI endpoint:", os.environ["LOCAL_NIM_URL"])

# from pathlib import Path
# from docker_storage import workshop_work_dir, glob_work_paths

# NB_DIR = Path.cwd().resolve()
# WORK_DIR = workshop_work_dir("M9-nvidia_nim")
# M7_NB = NB_DIR.parent / "M7-model_training"
# M7_WORK = workshop_work_dir("M7-model_training")
# _sft = glob_work_paths(M7_WORK, M7_NB, "sft_checkpoints/**/model/consolidated")
# M7_SFT = _sft[-1] if _sft else (M7_NB / "work" / "sft_checkpoints" / "sft-lora-final")
# MERGE_DIR = WORK_DIR / "merged_sft"
# print('M7 SFT LoRA:', (M7_SFT/'adapter_config.json').exists())


Enter your NGC API key:  ········


Docker login → nvcr.io …
Login Succeeded


CompletedProcess(args=['docker', 'login', 'nvcr.io', '-u', '$oauthtoken', '--password-stdin'], returncode=0)

## 2. Smoke-test catalog NIM (Nemotron Nano 9B)


## 3. Clean up

Stop and remove the NIM container(s) this notebook started, so the GPU memory and
ports are freed for the next module.

In [6]:
# --- Download fine-tuned checkpoint from Hugging Face (skip M7.3) ---
import os
import subprocess
from pathlib import Path

from docker_storage import workshop_download_dir
from notebook_env import ensure

ensure("huggingface_hub", ["huggingface-hub>=0.26.0"], quiet=True)

HF_TOKEN = "REDACTED_HF_TOKEN"
DOWNLOAD_DIR = workshop_download_dir("sft_checkpoint")

subprocess.run(["hf", "auth", "login", "--token", HF_TOKEN], check=True)

subprocess.run(
    [
        "hf", "download", "s4sarath/latest_sft_checkpoint",
        "--repo-type", "model",
        "--local-dir", str(DOWNLOAD_DIR),
    ],
    check=True,
)

# Next cell expects a consolidated checkpoint dir (M7 layout or flat HF export).
_consolidated = sorted(DOWNLOAD_DIR.glob("**/model/consolidated"))
_ckpt_dir = _consolidated[-1] if _consolidated else DOWNLOAD_DIR
os.environ["CUSTOM_MODEL_DIR"] = str(_ckpt_dir)

print("Downloaded checkpoint:", _ckpt_dir)
print("Files:", [p.name for p in sorted(_ckpt_dir.iterdir())[:12]], "…" if len(list(_ckpt_dir.iterdir())) > 12 else "")

ok: huggingface_hub


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `Nvidia-Eval` has been saved to /ephemeral/cache/hf/stored_tokens
Your token has been saved to /ephemeral/cache/hf/token
Login successful.
The current active token is: `Nvidia-Eval`


/ephemeral/downloads/sft_checkpoint
Downloaded checkpoint: /ephemeral/downloads/sft_checkpoint
Files: ['.cache', '.gitattributes', 'chat_template.jinja', 'config.json', 'configuration_nemotron_h.py', 'generation_config.json', 'model-00001-of-00013.safetensors', 'model-00002-of-00013.safetensors', 'model-00003-of-00013.safetensors', 'model-00004-of-00013.safetensors', 'model-00005-of-00013.safetensors', 'model-00006-of-00013.safetensors'] …


Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 1580.78it/s]


## 4. Deploy a custom model (Model-Free NIM) from your fine-tuned checkpoint

The catalog NIM above serves a *pre-packaged* model. To serve **your own** trained
checkpoint (e.g. the M7 SFT consolidated model), use a **Model-Free NIM**, which
loads weights from a directory you mount into the container.

Mirrors `gsi-training/9.custom_model_deployment/README.md`, scaled to **1 GPU
(`--tensor-parallel-size 1`)**. Run the HF download cell above if you skipped M7.3.

In [7]:
from pathlib import Path
from docker_storage import workshop_work_dir, glob_work_paths

LOCAL_NIM_CACHE = Path(os.environ["LOCAL_NIM_CACHE"])

NB_DIR = Path.cwd().resolve()
WORK_DIR = workshop_work_dir("M9-nvidia_nim")
M7_NB = NB_DIR.parent / "M7-model_training"
M7_WORK = workshop_work_dir("M7-model_training")



# --- Custom model: deploy your fine-tuned checkpoint behind a Model-Free NIM ---
# Nemotron-H SFT checkpoint via model-free-nim (vLLM), 1 GPU (TP=1).
# Requires NVIDIA driver >= 580 (CUDA 13) for image tag 2.0.5.

# Workshop SFT checkpoint on spacious volume (fallback: CUSTOM_MODEL_DIR from HF download cell).
_sft_ckpt = glob_work_paths(M7_WORK, M7_NB, "sft_checkpoints/**/model/consolidated")
CUSTOM_MODEL_DIR = str(_sft_ckpt[-1]) if _sft_ckpt else os.environ.get("CUSTOM_MODEL_DIR", "")

CUSTOM_NIM_IMAGE     = "nvcr.io/nim/nvidia/model-free-nim:2.0.5"
CUSTOM_NIM_CONTAINER = "workshop-custom-nim"
CUSTOM_NIM_PORT      = 8088
CUSTOM_SERVED_NAME   = "aml-custom-task-nim-1"
CUSTOM_NIM_MOUNT     = "/model"  # container path for checkpoint (read-only)

assert CUSTOM_MODEL_DIR and Path(CUSTOM_MODEL_DIR).is_dir(), (
    f"No SFT checkpoint found on spacious volume and CUSTOM_MODEL_DIR not set (got {CUSTOM_MODEL_DIR!r}). "
    "Run the HF download cell above, M7.3 sft.ipynb, or export CUSTOM_MODEL_DIR."
)

# model-free-nim:2.0.5 bundles PyTorch+cu130 — needs host driver >= 580.
_drv = subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip().split("\n")[0]
_drv_major = int(_drv.split(".")[0]) if _drv else 0
# if _drv_major < 580:
#     raise RuntimeError(
#         f"NVIDIA driver {_drv} is too old for model-free-nim:2.0.5 (needs >= 580). "
#         "On the host: sudo apt install -y nvidia-driver-580 && sudo reboot"
#     )

print("Checkpoint:", CUSTOM_MODEL_DIR)
print("Pulling Model-Free NIM image (first time: several minutes) …")
subprocess.check_call(["docker", "pull", CUSTOM_NIM_IMAGE])

subprocess.run(["docker", "rm", "-f", CUSTOM_NIM_CONTAINER], capture_output=True)

print("Starting custom NIM (1 GPU, TP=1) …")
subprocess.check_call([
    "docker", "run", "-d",
    "--name", CUSTOM_NIM_CONTAINER,
    "--gpus", '"device=0"',
    "--ipc=host",
    "--shm-size=16g",
    "-p", f"{CUSTOM_NIM_PORT}:8000",              # container listens on 8000
    "-v", f"{CUSTOM_MODEL_DIR}:{CUSTOM_NIM_MOUNT}:ro",
    "-v", f"{LOCAL_NIM_CACHE}:/opt/nim/.cache",
    "-e", f"NIM_MODEL_PATH={CUSTOM_NIM_MOUNT}",
    "-e", f"NIM_SERVED_MODEL_NAME={CUSTOM_SERVED_NAME}",
    "-e", "NIM_PASSTHROUGH_ARGS=--max-model-len 16000 --max-num-seqs 72",
    "-e", "NIM_MAX_NUM_SEQS=16",
    "-e", "NIM_GPU_MEMORY_UTILIZATION=0.95",
    "-e", "NIM_TRUST_CUSTOM_CODE=1",
    CUSTOM_NIM_IMAGE,
    "--tensor-parallel-size", "1",
    "--trust-remote-code",                         # required for Nemotron-H custom code
], env=os.environ.copy())

def _custom_ready():
    try:
        return requests.get(f"http://localhost:{CUSTOM_NIM_PORT}/v1/models", timeout=5).status_code == 200
    except Exception:
        return False

for _i in range(60):
    if _custom_ready():
        _r = requests.get(f"http://localhost:{CUSTOM_NIM_PORT}/v1/models", timeout=10)
        print("Custom NIM models:", [m["id"] for m in _r.json().get("data", [])])
        break
    print(f"  waiting for custom NIM … ({_i + 1}/60)")
    time.sleep(15)
else:
    raise RuntimeError(f"Custom NIM not ready — check: docker logs {CUSTOM_NIM_CONTAINER}")

# Smoke-test the custom endpoint.
r = requests.post(f"http://localhost:{CUSTOM_NIM_PORT}/v1/chat/completions", json={
    "model": CUSTOM_SERVED_NAME,
    "messages": [{"role": "user", "content": "In one sentence, what is AML structuring?"}],
    "max_tokens": 64,
}, timeout=120)
print("\nCustom model says:", r.json()["choices"][0]["message"]["content"])

Checkpoint: /ephemeral/downloads/sft_checkpoint
Pulling Model-Free NIM image (first time: several minutes) …
2.0.5: Pulling from nim/nvidia/model-free-nim
Digest: sha256:ad8d7db45c63fb2980fab2b580c539e7c10dfe5d636ee3a566bc8128e59be430
Status: Image is up to date for nvcr.io/nim/nvidia/model-free-nim:2.0.5
nvcr.io/nim/nvidia/model-free-nim:2.0.5
Starting custom NIM (1 GPU, TP=1) …
7e70ff680408be2ae97bafa56e982e88e04880daacb7756b6a083850e869040d
  waiting for custom NIM … (1/60)
  waiting for custom NIM … (2/60)
  waiting for custom NIM … (3/60)
  waiting for custom NIM … (4/60)
  waiting for custom NIM … (5/60)
Custom NIM models: ['aml-custom-task-nim-1']

Custom model says: AML structuring, also known as smurfing, is the practice of breaking down large amounts of cash or other assets into smaller, less noticeable transactions to avoid triggering regulatory reporting requirements or detection by financial institutions.


In [8]:
import requests

In [9]:
r = requests.post(f"http://localhost:{CUSTOM_NIM_PORT}/v1/chat/completions", json={
    "model": CUSTOM_SERVED_NAME,
    "messages": [{"role": "user", "content": "In one sentence, what is AML structuring?"}],
    "max_tokens": 64,
}, timeout=120)

In [10]:
r.json()["choices"][0]["message"]["content"]

'One sentence explanation for AML structuring: It is a deceptive method where individuals divide large money transfers into smaller amounts to evade detection by financial institutions and regulatory authorities.'